In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA = Path("data")
con = duckdb.connect()
train  = (DATA / "application_train.csv").as_posix()
bureau = (DATA / "bureau.csv").as_posix()

print(con.execute(f"""
    SELECT CREDIT_ACTIVE, COUNT(*) AS loans
    FROM read_csv_auto('{bureau}')
    GROUP BY CREDIT_ACTIVE ORDER BY loans DESC
""").df())

print(con.execute(f"""
    SELECT COUNT(*) AS applicants,
           ROUND(AVG(n), 1) AS avg_loans,
           MAX(n) AS max_loans
    FROM (SELECT SK_ID_CURR, COUNT(*) AS n
          FROM read_csv_auto('{bureau}') GROUP BY SK_ID_CURR)
""").df())

  CREDIT_ACTIVE    loans
0        Closed  1079273
1        Active   630607
2          Sold     6527
3      Bad debt       21
   applicants  avg_loans  max_loans
0      305811        5.6        116


In [2]:
bureau_features = con.execute(f"""
    SELECT SK_ID_CURR,
           COUNT(*)                                              AS bureau_loan_count,
           SUM(CASE WHEN CREDIT_ACTIVE = 'Active'   THEN 1 ELSE 0 END) AS bureau_active_count,
           SUM(CASE WHEN CREDIT_ACTIVE = 'Bad debt' THEN 1 ELSE 0 END) AS bureau_bad_debt_count,
           SUM(AMT_CREDIT_SUM_DEBT)                              AS bureau_total_debt,
           SUM(AMT_CREDIT_SUM_OVERDUE)                           AS bureau_total_overdue,
           MAX(CREDIT_DAY_OVERDUE)                               AS bureau_max_days_overdue,
           MIN(DAYS_CREDIT)                                      AS bureau_oldest_loan_days
    FROM read_csv_auto('{bureau}')
    GROUP BY SK_ID_CURR
""").df()

bureau_features.to_parquet(DATA / "bureau_features.parquet", index=False)
print(bureau_features.shape)
bureau_features.describe().T.round(1)

(305811, 8)


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,305811.0,278047.3,102849.6,100001.0,188878.5,277895.0,367184.5,456255.0
bureau_loan_count,305811.0,5.6,4.4,1.0,2.0,4.0,8.0,116.0
bureau_active_count,305811.0,2.1,1.8,0.0,1.0,2.0,3.0,32.0
bureau_bad_debt_count,305811.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
bureau_total_debt,297439.0,672319.9,1659897.5,-6981558.2,0.0,189747.0,700805.2,334498331.2
bureau_total_overdue,305811.0,212.8,15691.6,0.0,0.0,0.0,0.0,3756681.0
bureau_max_days_overdue,305811.0,4.4,85.6,0.0,0.0,0.0,0.0,2792.0
bureau_oldest_loan_days,305811.0,-1764.4,859.3,-2922.0,-2583.0,-1822.0,-1045.0,0.0


In [3]:
con.execute(f"""
    SELECT COUNT(*) AS loans_with_negative_debt,
           COUNT(DISTINCT SK_ID_CURR) AS people,
           MIN(AMT_CREDIT_SUM_DEBT) AS most_negative
    FROM read_csv_auto('{bureau}')
    WHERE AMT_CREDIT_SUM_DEBT < 0
""").df()

,loans_with_negative_debt,people,most_negative
0,8418,5886,-4705600.32


In [4]:
bureau_features = con.execute(f"""
    SELECT SK_ID_CURR,
           COUNT(*)                                                    AS bureau_loan_count,
           SUM(CASE WHEN CREDIT_ACTIVE = 'Active'   THEN 1 ELSE 0 END) AS bureau_active_count,
           SUM(CASE WHEN CREDIT_ACTIVE = 'Bad debt' THEN 1 ELSE 0 END) AS bureau_bad_debt_count,
           SUM(CASE WHEN AMT_CREDIT_SUM_DEBT < 0 THEN 0
                    ELSE AMT_CREDIT_SUM_DEBT END)                      AS bureau_total_debt,
           SUM(AMT_CREDIT_SUM_OVERDUE)                                 AS bureau_total_overdue,
           MAX(CREDIT_DAY_OVERDUE)                                     AS bureau_max_days_overdue,
           MIN(DAYS_CREDIT)                                            AS bureau_oldest_loan_days
    FROM read_csv_auto('{bureau}')
    GROUP BY SK_ID_CURR
""").df()

bureau_features.to_parquet(DATA / "bureau_features.parquet", index=False)
print(bureau_features.shape)
bureau_features.describe().T.round(1)

(305811, 8)


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,305811.0,278047.3,102849.6,100001.0,188878.5,277895.0,367184.5,456255.0
bureau_loan_count,305811.0,5.6,4.4,1.0,2.0,4.0,8.0,116.0
bureau_active_count,305811.0,2.1,1.8,0.0,1.0,2.0,3.0,32.0
bureau_bad_debt_count,305811.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
bureau_total_debt,297439.0,672540.9,1659775.8,0.0,0.0,189837.3,700980.8,334498331.2
bureau_total_overdue,305811.0,212.8,15691.6,0.0,0.0,0.0,0.0,3756681.0
bureau_max_days_overdue,305811.0,4.4,85.6,0.0,0.0,0.0,0.0,2792.0
bureau_oldest_loan_days,305811.0,-1764.4,859.3,-2922.0,-2583.0,-1822.0,-1045.0,0.0


In [5]:
prev = (DATA / "previous_application.csv").as_posix()

print(con.execute(f"""
    SELECT NAME_CONTRACT_STATUS, COUNT(*) AS applications
    FROM read_csv_auto('{prev}')
    GROUP BY 1 ORDER BY 2 DESC
""").df())

prev_features = con.execute(f"""
    SELECT SK_ID_CURR,
           COUNT(*)                                                             AS prev_app_count,
           SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END)   AS prev_approved_count,
           SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END)   AS prev_refused_count,
           ROUND(AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1.0 ELSE 0 END), 3) AS prev_refused_rate,
           AVG(AMT_CREDIT)                                                      AS prev_avg_credit,
           MAX(DAYS_DECISION)                                                   AS prev_last_decision_days
    FROM read_csv_auto('{prev}')
    GROUP BY SK_ID_CURR
""").df()

prev_features.to_parquet(DATA / "prev_features.parquet", index=False)
print(prev_features.shape)
prev_features.describe().T.round(2)

  NAME_CONTRACT_STATUS  applications
0             Approved       1036781
1             Canceled        316319
2              Refused        290678
3         Unused offer         26436
(338857, 7)


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,338857.0,278149.91,102879.19,100001.0,189061.00,278221.0,367302.0,456255.0
prev_app_count,338857.0,4.93,4.22,1.0,2.00,4.0,7.0,77.0
prev_approved_count,338857.0,3.06,2.14,0.0,1.00,3.0,4.0,27.0
prev_refused_count,338857.0,0.86,1.83,0.0,0.00,0.0,1.0,68.0
prev_refused_rate,338857.0,0.11,0.18,0.0,0.00,0.0,0.2,1.0
prev_avg_credit,338857.0,170331.85,164980.97,0.0,67416.75,118165.5,216412.5,4050000.0
prev_last_decision_days,338857.0,-481.49,536.93,-2922.0,-596.00,-304.0,-151.0,-1.0


In [6]:
inst = (DATA / "installments_payments.csv").as_posix()

con.execute(f"""
    SELECT
      MAX(DAYS_INSTALMENT)                              AS max_due_day,
      MAX(DAYS_ENTRY_PAYMENT)                           AS max_paid_day,
      COUNT(*) FILTER (WHERE DAYS_INSTALMENT > 0)       AS future_due_rows,
      COUNT(*) FILTER (WHERE DAYS_ENTRY_PAYMENT > 0)    AS future_paid_rows,
      COUNT(*) FILTER (WHERE DAYS_ENTRY_PAYMENT IS NULL) AS missing_paid_day
    FROM read_csv_auto('{inst}')
""").df()

,max_due_day,max_paid_day,future_due_rows,future_paid_rows,missing_paid_day
0,-1.0,-1.0,0,0,2905


In [7]:
inst_features = con.execute(f"""
    SELECT SK_ID_CURR,
           COUNT(*)                                                     AS inst_count,
           AVG(CASE WHEN DAYS_ENTRY_PAYMENT IS NULL THEN NULL
                    WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1.0
                    ELSE 0 END)                                         AS inst_late_rate,
           MAX(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT
                    THEN DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT ELSE 0 END) AS inst_max_days_late,
           AVG(CASE WHEN AMT_PAYMENT IS NULL THEN NULL
                    WHEN AMT_PAYMENT < AMT_INSTALMENT THEN 1.0
                    ELSE 0 END)                                         AS inst_underpaid_rate
    FROM read_csv_auto('{inst}')
    GROUP BY SK_ID_CURR
""").df()

inst_features.to_parquet(DATA / "inst_features.parquet", index=False)
print(inst_features.shape)
inst_features.describe().T.round(3)

(339587, 5)


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,339587.0,278154.892,102880.493,100001.0,189042.5,278238.000,367315.500,456255.0
inst_count,339587.0,40.065,41.053,1.0,12.0,25.000,51.000,372.0
inst_late_rate,339578.0,0.074,0.115,0.0,0.0,0.018,0.110,1.0
inst_max_days_late,339587.0,17.844,108.274,0.0,0.0,1.000,9.000,2884.0
inst_underpaid_rate,339578.0,0.083,0.148,0.0,0.0,0.000,0.105,1.0


In [8]:
model_table = con.execute(f"""
    SELECT a.*,
           b.* EXCLUDE (SK_ID_CURR),
           p.* EXCLUDE (SK_ID_CURR),
           i.* EXCLUDE (SK_ID_CURR)
    FROM read_csv_auto('{train}') a
    LEFT JOIN read_parquet('{(DATA / "bureau_features.parquet").as_posix()}') b USING (SK_ID_CURR)
    LEFT JOIN read_parquet('{(DATA / "prev_features.parquet").as_posix()}')   p USING (SK_ID_CURR)
    LEFT JOIN read_parquet('{(DATA / "inst_features.parquet").as_posix()}')   i USING (SK_ID_CURR)
""").df()

model_table.to_parquet(DATA / "model_table.parquet", index=False)
print("Shape:", model_table.shape)
print("Unique applicants:", model_table["SK_ID_CURR"].nunique())
print("Default rate:", round(100 * model_table["TARGET"].mean(), 2), "%")

Shape: (307511, 139)
Unique applicants: 307511
Default rate: 8.07 %


## Stage 4 findings: SQL history features

- Summarised **3 history tables** (~17.0M rows) into **17 per-applicant features** with DuckDB SQL.
- **bureau** (other lenders): loan counts, active/bad-debt counts, total debt, overdue amounts. 0.49% of loans had negative debt → set to 0 (documented assumption).
- **previous_application** (Home Credit): application count, approvals, refusals, refusal rate. Average refusal rate 11%.
- **installments_payments**: late-payment rate, max days late, underpaid rate. Average late rate 7.4%.
- **Leakage checks passed:** all history dates are before the application date (max = −1 day).
- **Final model table:** 307,511 rows × 139 columns, one row per applicant, default rate unchanged at 8.07%.
- Skipped for now: `POS_CASH_balance`, `credit_card_balance` (28% coverage), `bureau_balance`.